# GROMACS MD Simulation on Google Colab (Free T4 GPU)

**Goal:** Run 10 ns MD simulation using free T4 GPU  
**Speedup:** 5-10× faster than Mac M1 Pro CPU  
**Time:** ~2-3 hours total (including setup)  
**Cost:** FREE!

---

## ⚠️ CRITICAL: Enable GPU First!

1. Click **Runtime** → **Change runtime type**
2. **Hardware accelerator** → **GPU** (T4)
3. Click **Save**

---

## Why Compile from Source?

**Important:** Pre-built conda packages do NOT include GPU support!  
([Source: GROMACS Forums](https://gromacs.bioexcel.eu/t/gromacs-install-with-conda-but-cant-run-on-gpu/10856))

We must compile GROMACS with `-DGMX_GPU=CUDA` flag for GPU acceleration.  
This takes ~20-30 minutes but is the **only reliable method**.

---

## Timeline

| Step | Time | Description |
|------|------|-------------|
| 1. Verify GPU | 10 sec | Check T4 is available |
| 2. Install dependencies | 2-3 min | cmake, build tools |
| 3. Compile GROMACS | 20-30 min | Build with CUDA |
| 4. Mount Drive | 30 sec | Access your files |
| 5. Upload/verify files | 1-2 min | Get simulation files |
| 6. Run simulation | 1-2 hours | The actual MD |
| 7. Save results | 5 min | Copy to Drive |

**Total: ~2-3 hours** (vs 12+ hours on CPU!)

---


## Step 1: Verify GPU is Available

**Time:** 10 seconds

Must see "Tesla T4" or similar NVIDIA GPU.

In [1]:
# Check NVIDIA GPU
!nvidia-smi

print("\n" + "="*70)

import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,compute_cap', 
                        '--format=csv,noheader'], capture_output=True, text=True)

if result.returncode == 0:
    gpu_info = result.stdout.strip()
    print(f"\n✅ GPU Detected: {gpu_info}")
    
    # Check CUDA version
    cuda_result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
    if cuda_result.returncode == 0:
        for line in cuda_result.stdout.split('\n'):
            if 'release' in line:
                print(f"✅ CUDA: {line.strip()}")
    
    print("\n🎉 GPU ready! Proceed to Step 2.")
else:
    print("\n❌ ERROR: No GPU detected!")
    print("\n⚠️  FIX THIS NOW:")
    print("   1. Runtime → Change runtime type")
    print("   2. Hardware accelerator → GPU")
    print("   3. Save → Re-run this cell")

print("="*70)

: 

## Step 2: Install Build Dependencies

**Time:** 2-3 minutes

Install cmake, compilers, and libraries needed to build GROMACS.

In [2]:
%%time
print("📦 Installing build dependencies...\n")

# Update package list
!apt-get update -qq

# Install required packages
!apt-get install -y -qq \
    cmake \
    build-essential \
    libfftw3-dev \
    libopenmpi-dev \
    openmpi-bin \
    > /dev/null 2>&1

# Verify cmake version
print("\n📋 Checking installed tools:")
!cmake --version | head -1
!gcc --version | head -1
!nvcc --version | grep release

print("\n✅ Dependencies installed!")

## Step 3: Download and Compile GROMACS with CUDA

**Time:** 20-30 minutes

This is the critical step that enables GPU acceleration.  
We compile GROMACS 2024.4 with `-DGMX_GPU=CUDA` flag.

☕ **Go get a coffee** - this takes a while but only needs to run once per session.

In [ ]:
%%time
import os

# GROMACS version to install
GMX_VERSION = "2024.4"
GMX_TARBALL = f"gromacs-{GMX_VERSION}.tar.gz"
GMX_URL = f"https://ftp.gromacs.org/gromacs/{GMX_TARBALL}"

print(f"🧬 Building GROMACS {GMX_VERSION} with CUDA GPU support")
print(f"   This takes 20-30 minutes. Please wait...\n")
print("="*70)

# Create build directory
!mkdir -p /content/gromacs_build
os.chdir('/content/gromacs_build')

# Download GROMACS source
print("\n📥 Step 1/4: Downloading GROMACS source...")
!wget -q {GMX_URL}
!tar -xzf {GMX_TARBALL}
print("   ✅ Download complete")

# Create build directory
build_dir = f'/content/gromacs_build/gromacs-{GMX_VERSION}/build'
!mkdir -p {build_dir}
os.chdir(build_dir)

# Configure with cmake
print("\n⚙️  Step 2/4: Configuring with cmake (CUDA enabled)...")
!cmake .. \
    -DGMX_BUILD_OWN_FFTW=ON \
    -DGMX_GPU=CUDA \
    -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda \
    -DGMX_CUDA_TARGET_COMPUTE="70;75;80;86" \
    -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs \
    -DREGRESSIONTEST_DOWNLOAD=OFF \
    > /tmp/cmake_output.log 2>&1

# Check if cmake succeeded
if os.path.exists('Makefile'):
    print("   ✅ Configuration complete")
else:
    print("   ❌ Configuration FAILED! Check /tmp/cmake_output.log")
    !tail -50 /tmp/cmake_output.log

# Compile (this is the slow part)
print("\n🔨 Step 3/4: Compiling GROMACS (15-25 minutes)...")
print("   Using all available CPU cores...")
!make -j$(nproc) > /tmp/make_output.log 2>&1

# Check if make succeeded
if os.path.exists('bin/gmx'):
    print("   ✅ Compilation complete")
else:
    print("   ❌ Compilation FAILED! Check /tmp/make_output.log")
    !tail -50 /tmp/make_output.log

# Install
print("\n📦 Step 4/4: Installing GROMACS...")
!make install > /dev/null 2>&1
print("   ✅ Installation complete")

# Add to PATH
os.environ['PATH'] = f"/usr/local/gromacs/bin:{os.environ['PATH']}"
os.environ['GMX_MAXBACKUP'] = '-1'

print("\n" + "="*70)
print("🎉 GROMACS compiled successfully with CUDA support!")
print("="*70)

## Step 3b: Verify GPU Support is Enabled

**Time:** 10 seconds

**CRITICAL:** Verify GROMACS was compiled with CUDA support.

In [ ]:
import os
import subprocess

# Make sure PATH is set
os.environ['PATH'] = f"/usr/local/gromacs/bin:{os.environ['PATH']}"

print("📋 GROMACS Version and GPU Support:\n")

# Get version info
result = subprocess.run(['/usr/local/gromacs/bin/gmx', '--version'], 
                       capture_output=True, text=True)
output = result.stdout

# Print key lines
important_keys = ['GROMACS version', 'GPU support', 'CUDA', 'OpenCL', 'SIMD']
for line in output.split('\n'):
    for key in important_keys:
        if key in line:
            print(f"  {line.strip()}")
            break

print("\n" + "="*70)

# Check for CUDA support
if 'CUDA' in output and ('enabled' in output.lower() or 'GPU support' in output):
    print("✅ SUCCESS: GROMACS has CUDA GPU support enabled!")
    print("   Ready to run simulations with GPU acceleration.")
else:
    print("⚠️  WARNING: CUDA support may not be enabled.")
    print("   Check full output above.")
    print("\nFull version output:")
    print(output)

print("="*70)

## Step 4: Mount Google Drive

**Time:** 30 seconds

Click the authorization link when prompted.

In [ ]:
from google.colab import drive

print("📁 Mounting Google Drive...\n")
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("   Your files are at: /content/drive/MyDrive/")

## Step 5: Upload Simulation Files

**Time:** 1-2 minutes

### Option A: Upload directly to Colab (EASIEST)
1. Click the **folder icon** in left sidebar
2. Click **upload icon** (arrow pointing up)
3. Upload these files:
   - `md.tpr` (9.5 MB)
   - `md.cpt` (8.5 MB) - checkpoint file
   - `topol.top` (873 KB)

### Option B: Upload to Google Drive first
1. Go to https://drive.google.com
2. Create folder: `p53_simulation`
3. Upload files there
4. Run cell below to copy them

In [ ]:
import os

# Create working directory
work_dir = '/content/md_simulation'
!mkdir -p {work_dir}

print("📁 Setting up simulation files...\n")

# Check Option A: Files uploaded directly to Colab
colab_files = ['/content/md.tpr', '/content/md.cpt', '/content/topol.top']
colab_found = all(os.path.exists(f) for f in colab_files)

# Check Option B: Files in Google Drive
drive_dir = '/content/drive/MyDrive/p53_simulation'
drive_files = [f'{drive_dir}/md.tpr', f'{drive_dir}/md.cpt', f'{drive_dir}/topol.top']
drive_found = all(os.path.exists(f) for f in drive_files)

if colab_found:
    print("✅ Found files uploaded to Colab")
    !cp /content/md.tpr /content/md.cpt /content/topol.top {work_dir}/
    print("   Copied to working directory.")
elif drive_found:
    print("✅ Found files in Google Drive")
    !cp {drive_dir}/* {work_dir}/
    print("   Copied to working directory.")
else:
    print("❌ Files not found!\n")
    print("📥 Please upload your files using ONE of these methods:\n")
    print("   Option A (Easiest): Upload directly to Colab")
    print("   - Click folder icon (📁) in left sidebar")
    print("   - Click upload icon (⬆️)")
    print("   - Select: md.tpr, md.cpt, topol.top\n")
    print("   Option B: Upload to Google Drive")
    print(f"   - Create folder: {drive_dir}")
    print("   - Upload files there\n")
    print("   Then re-run this cell.")

# Verify files
print("\n" + "="*70)
print("📋 Files in working directory:")
!ls -lh {work_dir}/

# Check file sizes
required_files = {
    'md.tpr': (9.0, 10.0),
    'md.cpt': (8.0, 9.0),
    'topol.top': (0.8, 1.0)
}

all_ok = True
for filename, (min_mb, max_mb) in required_files.items():
    filepath = f'{work_dir}/{filename}'
    if os.path.exists(filepath):
        size_mb = os.path.getsize(filepath) / 1024 / 1024
        if size_mb < 0.1:  # File too small (probably LFS pointer)
            print(f"⚠️  {filename} is too small ({size_mb:.3f} MB) - may be corrupted!")
            all_ok = False
    else:
        all_ok = False

if all_ok:
    print("\n✅ All files ready! Proceed to Step 6.")
print("="*70)

## Step 6: Run MD Simulation with GPU 🚀

**Time:** 1-2 hours for 10 ns (or remaining from checkpoint)

**⚠️ IMPORTANT:**
- This runs for 1-2 hours
- Progress updates every 20 ps
- Don't close browser tab!
- Colab free tier: max 12-hour sessions

### Expected Performance
- **T4 GPU:** 150-250 ns/day
- **Mac M1 CPU:** 26 ns/day
- **Speedup:** 6-10×!

In [ ]:
%%time
import os

# Ensure PATH is set
os.environ['PATH'] = f"/usr/local/gromacs/bin:{os.environ['PATH']}"

# Change to working directory
work_dir = '/content/md_simulation'
os.chdir(work_dir)

# Check if we have a checkpoint (resuming) or starting fresh
has_checkpoint = os.path.exists('md.cpt')

print("🚀 Starting MD Simulation with T4 GPU")
print(f"📁 Working directory: {work_dir}")
print("\n" + "="*70)
print("SIMULATION CONFIGURATION")
print("="*70)

if has_checkpoint:
    print("Mode: RESUME from checkpoint")
    print("Starting from: Previous checkpoint (check log for exact step)")
else:
    print("Mode: FRESH START")
    print("Starting from: 0 ps")

print("Target: 10,000 ps (10 ns)")
print("Expected time: 1-2 hours on T4 GPU")
print("Expected performance: 150-250 ns/day")
print("\n" + "="*70)
print("LIVE PROGRESS (updates every 20 ps)")
print("="*70 + "\n")

# Build command
cmd = '/usr/local/gromacs/bin/gmx mdrun -v -deffnm md'
if has_checkpoint:
    cmd += ' -cpi md.cpt'
cmd += ' -ntomp 2 -nb gpu -pme gpu -bonded gpu -update gpu'

# Run simulation
!{cmd}

print("\n" + "="*70)
print("🎉 SIMULATION COMPLETE!")
print("="*70)

## Step 7: Verify Results

**Time:** 10 seconds

Check simulation completed successfully.

In [ ]:
import os

work_dir = '/content/md_simulation'

print("📊 Checking simulation results...\n")

# Check output files
output_files = ['md.xtc', 'md.log', 'md.edr', 'md.cpt']
for filename in output_files:
    filepath = f'{work_dir}/{filename}'
    if os.path.exists(filepath):
        size = os.path.getsize(filepath) / 1024 / 1024
        print(f"✅ {filename:12s} ({size:7.1f} MB)")
    else:
        print(f"❌ {filename:12s} MISSING")

# Check final step from log
print("\n📋 Final progress from log:")
!tail -100 {work_dir}/md.log | grep -E "Step|time" | tail -5

print("\n⚡ Performance achieved:")
!grep 'Performance' {work_dir}/md.log | tail -3

# Check for successful completion
print("\n📋 Completion status:")
!grep -i 'finished' {work_dir}/md.log || echo "⚠️  'Finished' not found - check manually"

print("\n" + "="*70)

## Step 8: Save Results to Google Drive

**Time:** 5-10 minutes

**IMPORTANT:** Save before Colab disconnects!

In [ ]:
%%time
import os
from datetime import datetime

work_dir = '/content/md_simulation'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = f'/content/drive/MyDrive/p53_md_results_{timestamp}'

print(f"📦 Saving results to Google Drive...\n")
print(f"   Destination: {results_dir}\n")

# Create results directory
!mkdir -p {results_dir}

# Copy all important files
files_to_save = ['md.xtc', 'md.log', 'md.edr', 'md.cpt', 'md.tpr', 'topol.top']
for filename in files_to_save:
    src = f'{work_dir}/{filename}'
    if os.path.exists(src):
        !cp {src} {results_dir}/
        size = os.path.getsize(src) / 1024 / 1024
        print(f"   ✅ Saved {filename:12s} ({size:7.1f} MB)")
    else:
        print(f"   ⚠️  {filename} not found")

# Create summary file
summary_path = f'{results_dir}/SUMMARY.txt'
with open(summary_path, 'w') as f:
    f.write("p53 MD Simulation Results\n")
    f.write("="*50 + "\n")
    f.write(f"Completed: {timestamp}\n")
    f.write(f"Platform: Google Colab (T4 GPU)\n")
    f.write(f"Simulation: A189S_M133L_S95T\n")
    f.write(f"\nFiles saved:\n")
    for filename in files_to_save:
        filepath = f'{work_dir}/{filename}'
        if os.path.exists(filepath):
            size = os.path.getsize(filepath) / 1024 / 1024
            f.write(f"  - {filename}: {size:.1f} MB\n")

print(f"\n✅ Results saved to Google Drive!")
print(f"\n📁 Location: {results_dir}")
print("\n📥 To download:")
print("   1. Go to: https://drive.google.com")
print(f"   2. Find folder: p53_md_results_{timestamp}")
print("   3. Right-click → Download (or select all files)")
print("\n" + "="*70)

---

## 🔄 Emergency: Save Checkpoint (If Timeout Approaching)

**Run this if:**
- Session will timeout (approaching 12 hours)
- Need to stop early
- Want to resume in new session

**How to use:**
1. Click **stop** (⏹️) on simulation cell
2. Wait 30 seconds
3. Run this cell

In [ ]:
import os
from datetime import datetime

work_dir = '/content/md_simulation'
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
checkpoint_dir = f'/content/drive/MyDrive/p53_checkpoint_{timestamp}'

print(f"💾 Saving checkpoint for resume...\n")

!mkdir -p {checkpoint_dir}

# Copy checkpoint files
!cp {work_dir}/md.cpt {checkpoint_dir}/ 2>/dev/null || echo "No checkpoint file"
!cp {work_dir}/md.tpr {checkpoint_dir}/
!cp {work_dir}/topol.top {checkpoint_dir}/
!cp {work_dir}/md.log {checkpoint_dir}/md_partial.log 2>/dev/null || echo "No log yet"
!cp {work_dir}/md.xtc {checkpoint_dir}/md_partial.xtc 2>/dev/null || echo "No trajectory yet"

# Show current progress
print("\n📊 Checkpoint saved at:")
!tail -10 {work_dir}/md.log 2>/dev/null | grep -E 'Step|time' | tail -2 || echo "Check log manually"

print(f"\n✅ Checkpoint saved to: {checkpoint_dir}")
print("\n🔄 To resume in new session:")
print("   1. Start new Colab session")
print("   2. Re-run Steps 1-4 (GPU check, compile, mount Drive)")
print(f"   3. Copy files from: {checkpoint_dir}")
print("   4. Run simulation with -cpi md.cpt flag")

---

## 🐛 Troubleshooting

### Error: "No GPU available"
```
Runtime → Change runtime type → GPU → Save
Then restart runtime and re-run all cells.
```

### Error: "gmx: command not found"
```
Re-run Step 3 (compile GROMACS).
Make sure compilation completed successfully.
```

### Error: "File not found: md.tpr"
```
1. Check files were uploaded correctly
2. Verify file sizes (md.tpr ~9 MB, not 1 KB)
3. If using Git LFS, files may be pointer files
```

### Error: "LINCS warning" or "NaN detected"
```
Checkpoint file may be corrupted.
Re-upload md.cpt from your Mac.
```

### Compilation fails at cmake
```
Run: !cat /tmp/cmake_output.log | tail -100
Look for error messages.
Common fix: Update cmake or check CUDA paths.
```

### Performance slower than expected
```
Expected: 150-250 ns/day on T4

Check GPU utilization:
!nvidia-smi

GPU-Util should be >90%.
If low, try reducing -ntomp threads.
```

### Session disconnects during simulation
```
1. Run checkpoint save cell above
2. Start new Colab session
3. Re-run Steps 1-5
4. Resume from checkpoint

Tip: Colab Pro ($10/month) has 24-hour sessions.
```

---

## 📊 Performance Comparison

| Platform | Hardware | Performance | 10 ns Time |
|----------|----------|-------------|------------|
| Mac | M1 Pro CPU | 26 ns/day | 12+ hours |
| **Colab** | **T4 GPU** | **150-250 ns/day** | **1-2 hours** |
| Windows | RTX 3060 | 150-250 ns/day | 1-2 hours |

**Speedup: 6-10× faster!** 🚀

---

## ✅ Success Checklist

**Before simulation:**
- [ ] GPU shows T4 (Step 1)
- [ ] GROMACS compiled with CUDA (Step 3b)
- [ ] Files uploaded and correct size (Step 5)

**During simulation:**
- [ ] Progress updates showing
- [ ] Performance 150-250 ns/day
- [ ] No errors in output

**After simulation:**
- [ ] md.xtc is 500-1000 MB
- [ ] Log shows "Finished mdrun"
- [ ] Results saved to Drive

---

## 📚 Sources

- [GROMACS Installation Guide](https://manual.gromacs.org/current/install-guide/index.html)
- [GROMACS Conda GPU Issue](https://gromacs.bioexcel.eu/t/gromacs-install-with-conda-but-cant-run-on-gpu/10856)
- [bioinfkaustin/gromacs-on-colab](https://github.com/bioinfkaustin/gromacs-on-colab)
- [TheBiomics GROMACS GPU Guide](https://www.thebiomics.com/research/gromacs-installation-with-gpu.html)

---

*Created: 2026-01-27*  
*Project: p53 StabiliMut Initiative 2*  
*GROMACS: 2024.4 with CUDA*  
*Platform: Google Colab (Free T4 GPU)*
